In [1]:
# ============================================
# SECTION 03 - DATA ENGINEERING CHALLENGES
# Expernetic Data Engineering Assignment
# Candidate: Ishini Ellewela
# Dataset: Inside Airbnb - Bangkok, Thailand
# ============================================

import pandas as pd
import numpy as np
import os
import sqlite3
import json
from datetime import datetime

# Paths
data_path = r"C:\Users\Ishini\Desktop\Expernetic_Assignment\data"
charts_path = r"C:\Users\Ishini\Desktop\Expernetic_Assignment\charts"

print("Libraries imported successfully!")
print(f"SQLite version: {sqlite3.sqlite_version}")

Libraries imported successfully!
SQLite version: 3.50.4


In [2]:
# ============================================
# STEP 1: DATA INGESTION & PROFILING
# ============================================

print("Loading datasets...\n")

listings = pd.read_csv(os.path.join(data_path, "listings.csv.gz"), compression='gzip')
reviews = pd.read_csv(os.path.join(data_path, "reviews.csv.gz"), compression='gzip')
calendar = pd.read_csv(os.path.join(data_path, "calendar.csv.gz"), compression='gzip')
neighbourhoods = pd.read_csv(os.path.join(data_path, "neighbourhoods.csv"))

print("=== DATA INGESTION REPORT ===")
datasets = {
    'listings': listings,
    'reviews': reviews,
    'calendar': calendar,
    'neighbourhoods': neighbourhoods
}

for name, df in datasets.items():
    null_pct = df.isnull().mean().mean() * 100
    print(f"\n{name.upper()}")
    print(f"  Rows:            {df.shape[0]:>10,}")
    print(f"  Columns:         {df.shape[1]:>10}")
    print(f"  Memory usage:    {df.memory_usage(deep=True).sum()/1024/1024:>9.1f} MB")
    print(f"  Avg missing %:   {null_pct:>9.1f}%")
    print(f"  Duplicate rows:  {df.duplicated().sum():>10,}")

Loading datasets...

=== DATA INGESTION REPORT ===

LISTINGS
  Rows:                31,069
  Columns:                 90
  Memory usage:        155.9 MB
  Avg missing %:        20.8%
  Duplicate rows:           0

REVIEWS
  Rows:               693,647
  Columns:                  6
  Memory usage:        329.8 MB
  Avg missing %:         0.0%
  Duplicate rows:           0

CALENDAR
  Rows:            11,340,155
  Columns:                  5
  Memory usage:       1438.4 MB
  Avg missing %:         0.0%
  Duplicate rows:           0

NEIGHBOURHOODS
  Rows:                    50
  Columns:                  2
  Memory usage:          0.0 MB
  Avg missing %:        50.0%
  Duplicate rows:           0


In [4]:
# ============================================
# STEP 2: DATA CLEANING & STANDARDIZATION
# ============================================

print("=== CLEANING PIPELINE ===\n")

# --- LISTINGS CLEANING ---
listings_clean = listings.copy()

# 1. Clean price column
listings_clean['price_clean'] = (
    listings_clean['price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)
print(f"✓ Price cleaned: converted from string to float")

# 2. Parse date columns
date_cols = ['host_since', 'first_review', 'last_review', 'last_scraped']
for col in date_cols:
    if col in listings_clean.columns:
        listings_clean[col] = pd.to_datetime(listings_clean[col], errors='coerce')
print(f"✓ Date columns parsed: {date_cols}")

# 3. Clean boolean columns
bool_cols = ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'instant_bookable']
for col in bool_cols:
    if col in listings_clean.columns:
        listings_clean[col] = listings_clean[col].map({'t': True, 'f': False})
print(f"✓ Boolean columns standardized: {bool_cols}")

# 4. Check percentage columns — already float in this dataset
pct_cols = ['host_response_rate', 'host_acceptance_rate']
for col in pct_cols:
    if col in listings_clean.columns:
        print(f"  {col}: dtype={listings_clean[col].dtype}, already numeric — no cleaning needed")
print(f"✓ Percentage columns verified as numeric")

# 5. Remove invalid prices
before = len(listings_clean)
listings_clean = listings_clean[
    (listings_clean['price_clean'] > 0) &
    (listings_clean['price_clean'] < 10000)
].copy()
after = len(listings_clean)
print(f"✓ Invalid prices removed: {before - after:,} rows removed")

# --- CALENDAR CLEANING ---
calendar_clean = calendar.copy()
calendar_clean['date'] = pd.to_datetime(calendar_clean['date'])
calendar_clean['available'] = calendar_clean['available'].map({'t': True, 'f': False})
print(f"✓ Calendar dates parsed and availability standardized")

# --- REVIEWS CLEANING ---
reviews_clean = reviews.copy()
reviews_clean['date'] = pd.to_datetime(reviews_clean['date'])
reviews_clean['comments'] = reviews_clean['comments'].fillna('')
print(f"✓ Reviews dates parsed")

print(f"\n=== CLEANING SUMMARY ===")
print(f"Listings after cleaning: {len(listings_clean):,}")
print(f"Calendar after cleaning: {len(calendar_clean):,}")
print(f"Reviews after cleaning:  {len(reviews_clean):,}")

=== CLEANING PIPELINE ===

✓ Price cleaned: converted from string to float
✓ Date columns parsed: ['host_since', 'first_review', 'last_review', 'last_scraped']
✓ Boolean columns standardized: ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'instant_bookable']
  host_response_rate: dtype=float64, already numeric — no cleaning needed
  host_acceptance_rate: dtype=float64, already numeric — no cleaning needed
✓ Percentage columns verified as numeric
✓ Invalid prices removed: 2,865 rows removed
✓ Calendar dates parsed and availability standardized
✓ Reviews dates parsed

=== CLEANING SUMMARY ===
Listings after cleaning: 28,204
Calendar after cleaning: 11,340,155
Reviews after cleaning:  693,647


In [5]:
# ============================================
# STEP 3: DATA ENRICHMENT & JOINING
# ============================================

print("=== JOINING DATASETS ===\n")

# 1. Review summary per listing
review_summary = reviews_clean.groupby('listing_id').agg(
    total_reviews=('id', 'count'),
    first_review_date=('date', 'min'),
    last_review_date=('date', 'max'),
    avg_comment_length=('comments', lambda x: x.str.len().mean())
).reset_index()

print(f"✓ Review summary created: {len(review_summary):,} listings with reviews")

# 2. Calendar summary per listing
calendar_summary = calendar_clean.groupby('listing_id').agg(
    total_available_days=('available', 'sum'),
    total_booked_days=('available', lambda x: (~x).sum()),
    occupancy_rate=('available', lambda x: (~x).mean() * 100)
).reset_index()

print(f"✓ Calendar summary created: {len(calendar_summary):,} listings")

# 3. Join listings with review summary
enriched_listings = listings_clean.merge(
    review_summary,
    left_on='id',
    right_on='listing_id',
    how='left'
)
print(f"✓ Listings joined with review summary: {len(enriched_listings):,} rows")

# 4. Join with calendar summary
enriched_listings = enriched_listings.merge(
    calendar_summary,
    left_on='id',
    right_on='listing_id',
    how='left'
)
print(f"✓ Listings joined with calendar summary: {len(enriched_listings):,} rows")

# 5. Join with neighbourhoods
enriched_listings = enriched_listings.merge(
    neighbourhoods,
    left_on='neighbourhood_cleansed',
    right_on='neighbourhood',
    how='left'
)
print(f"✓ Listings joined with neighbourhoods: {len(enriched_listings):,} rows")

print(f"\n=== ENRICHED DATASET ===")
print(f"Final rows: {len(enriched_listings):,}")
print(f"Final columns: {enriched_listings.shape[1]}")
print(f"\nNew columns added:")
print(f"  - total_reviews (from reviews)")
print(f"  - first/last_review_date (from reviews)")
print(f"  - avg_comment_length (from reviews)")
print(f"  - total_available_days (from calendar)")
print(f"  - total_booked_days (from calendar)")
print(f"  - occupancy_rate (from calendar)")

=== JOINING DATASETS ===

✓ Review summary created: 20,816 listings with reviews
✓ Calendar summary created: 31,069 listings
✓ Listings joined with review summary: 28,204 rows
✓ Listings joined with calendar summary: 28,204 rows
✓ Listings joined with neighbourhoods: 28,204 rows

=== ENRICHED DATASET ===
Final rows: 28,204
Final columns: 102

New columns added:
  - total_reviews (from reviews)
  - first/last_review_date (from reviews)
  - avg_comment_length (from reviews)
  - total_available_days (from calendar)
  - total_booked_days (from calendar)
  - occupancy_rate (from calendar)


In [6]:
# ============================================
# STEP 4: DATA QUALITY REPORT
# ============================================

print("=== DATA QUALITY REPORT ===\n")

key_columns = [
    'id', 'name', 'price_clean', 'room_type', 'neighbourhood_cleansed',
    'latitude', 'longitude', 'review_scores_rating', 'host_is_superhost',
    'availability_365', 'number_of_reviews', 'occupancy_rate'
]

print(f"{'Column':<35} {'Missing':<10} {'Missing %':<12} {'Unique Values':<15} {'Data Type'}")
print("-" * 90)

for col in key_columns:
    if col in enriched_listings.columns:
        missing = enriched_listings[col].isnull().sum()
        missing_pct = missing / len(enriched_listings) * 100
        unique = enriched_listings[col].nunique()
        dtype = str(enriched_listings[col].dtype)
        print(f"{col:<35} {missing:<10,} {missing_pct:<11.1f}% {unique:<15,} {dtype}")

print(f"\n=== VALIDATION CHECKS ===")
print(f"✓ Negative prices: {(enriched_listings['price_clean'] < 0).sum()}")
print(f"✓ Prices > 10,000: {(enriched_listings['price_clean'] > 10000).sum()}")
print(f"✓ Invalid coordinates: {((enriched_listings['latitude'] < 13) | (enriched_listings['latitude'] > 14)).sum()}")
print(f"✓ Duplicate listing IDs: {enriched_listings['id'].duplicated().sum()}")
print(f"✓ Review scores > 5: {(enriched_listings['review_scores_rating'] > 5).sum()}")
print(f"✓ Availability > 365: {(enriched_listings['availability_365'] > 365).sum()}")

=== DATA QUALITY REPORT ===

Column                              Missing    Missing %    Unique Values   Data Type
------------------------------------------------------------------------------------------
id                                  0          0.0        % 28,204          int64
name                                0          0.0        % 25,986          str
price_clean                         0          0.0        % 14,885          float64
room_type                           0          0.0        % 4               str
neighbourhood_cleansed              0          0.0        % 50              str
latitude                            0          0.0        % 19,647          float64
longitude                           0          0.0        % 20,337          float64
review_scores_rating                9,091      32.2       % 148             float64
host_is_superhost                   0          0.0        % 2               bool
availability_365                    0          0.0     

In [7]:
# ============================================
# STEP 5: SQL QUERIES USING SQLITE
# ============================================

print("=== LOADING DATA INTO SQLITE ===\n")

# Create in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Load key tables
listings_clean[['id', 'name', 'neighbourhood_cleansed', 'room_type',
                 'price_clean', 'review_scores_rating', 'host_is_superhost',
                 'availability_365', 'number_of_reviews',
                 'calculated_host_listings_count']].to_sql('listings', conn, index=False, if_exists='replace')

reviews_clean[['listing_id', 'id', 'date', 'reviewer_id']].to_sql('reviews', conn, index=False, if_exists='replace')

calendar_summary.to_sql('calendar_summary', conn, index=False, if_exists='replace')

print("✓ Tables loaded into SQLite")
print("\nTables available: listings, reviews, calendar_summary")

=== LOADING DATA INTO SQLITE ===

✓ Tables loaded into SQLite

Tables available: listings, reviews, calendar_summary


In [8]:
# SQL QUERY 1: Average price by room type
print("=== QUERY 1: Average Price by Room Type ===")
q1 = pd.read_sql_query("""
    SELECT 
        room_type,
        COUNT(*) as total_listings,
        ROUND(AVG(price_clean), 0) as avg_price,
        ROUND(MIN(price_clean), 0) as min_price,
        ROUND(MAX(price_clean), 0) as max_price
    FROM listings
    GROUP BY room_type
    ORDER BY avg_price DESC
""", conn)
print(q1.to_string(index=False))

# SQL QUERY 2: Top 10 neighbourhoods by listing count and avg price
print("\n=== QUERY 2: Top 10 Neighbourhoods ===")
q2 = pd.read_sql_query("""
    SELECT 
        neighbourhood_cleansed,
        COUNT(*) as total_listings,
        ROUND(AVG(price_clean), 0) as avg_price,
        ROUND(AVG(review_scores_rating), 2) as avg_rating
    FROM listings
    WHERE review_scores_rating IS NOT NULL
    GROUP BY neighbourhood_cleansed
    ORDER BY total_listings DESC
    LIMIT 10
""", conn)
print(q2.to_string(index=False))

# SQL QUERY 3: Superhost performance comparison
print("\n=== QUERY 3: Superhost vs Regular Host Performance ===")
q3 = pd.read_sql_query("""
    SELECT 
        CASE WHEN host_is_superhost = 1 THEN 'Superhost' ELSE 'Regular Host' END as host_type,
        COUNT(*) as total_listings,
        ROUND(AVG(price_clean), 0) as avg_price,
        ROUND(AVG(review_scores_rating), 2) as avg_rating,
        ROUND(AVG(number_of_reviews), 1) as avg_reviews
    FROM listings
    GROUP BY host_is_superhost
""", conn)
print(q3.to_string(index=False))

# SQL QUERY 4: High value listings
print("\n=== QUERY 4: Top 10 Highest Rated & Most Reviewed Listings ===")
q4 = pd.read_sql_query("""
    SELECT 
        id,
        name,
        neighbourhood_cleansed,
        room_type,
        ROUND(price_clean, 0) as price,
        review_scores_rating as rating,
        number_of_reviews as reviews
    FROM listings
    WHERE review_scores_rating >= 4.9
    AND number_of_reviews >= 50
    ORDER BY number_of_reviews DESC
    LIMIT 10
""", conn)
print(q4.to_string(index=False))

# SQL QUERY 5: Commercial hosts
print("\n=== QUERY 5: Commercial Host Analysis ===")
q5 = pd.read_sql_query("""
    SELECT 
        CASE 
            WHEN calculated_host_listings_count = 1 THEN 'Single listing'
            WHEN calculated_host_listings_count BETWEEN 2 AND 5 THEN '2-5 listings'
            WHEN calculated_host_listings_count BETWEEN 6 AND 20 THEN '6-20 listings'
            ELSE '20+ listings'
        END as host_category,
        COUNT(*) as total_listings,
        ROUND(AVG(price_clean), 0) as avg_price,
        ROUND(AVG(review_scores_rating), 2) as avg_rating
    FROM listings
    GROUP BY host_category
    ORDER BY total_listings DESC
""", conn)
print(q5.to_string(index=False))

conn.close()
print("\n✓ All SQL queries complete. Database connection closed.")

=== QUERY 1: Average Price by Room Type ===
      room_type  total_listings  avg_price  min_price  max_price
Entire home/apt           18673     2179.0       13.0     9997.0
     Hotel room             261     2162.0      315.0     9999.0
   Private room            8774     1795.0       40.0     9999.0
    Shared room             496      599.0        1.0     5118.0

=== QUERY 2: Top 10 Neighbourhoods ===
neighbourhood_cleansed  total_listings  avg_price  avg_rating
               Vadhana            2971     2637.0        4.73
           Khlong Toei            2746     2185.0        4.70
           Huai Khwang            1923     2036.0        4.61
           Ratchathewi            1204     2115.0        4.67
                Sathon             906     1840.0        4.67
              Bang Rak             827     2493.0        4.69
          Phra Khanong             824     1790.0        4.72
           Phra Nakhon             778     1707.0        4.60
            Chatu Chak           

In [10]:
# ============================================
# STEP 6: SAVE ENRICHED DATASET
# ============================================

# Save enriched listings as CSV
output_path = os.path.join(data_path, "enriched_listings.csv")
enriched_listings.to_csv(output_path, index=False)
print(f"✓ Enriched dataset saved: {output_path}")
print(f"  Rows: {len(enriched_listings):,}")
print(f"  Columns: {enriched_listings.shape[1]}")

# Engineering decision log
print("\n=== ENGINEERING DECISION LOG ===")
print("""
DECISION 1 - SQLite for SQL layer:
  Options considered: PostgreSQL, DuckDB, SQLite
  Choice: SQLite (in-memory)
  Reason: No installation needed, portable, sufficient for this dataset size
  Trade-off: Cannot handle concurrent writes, not production-grade

DECISION 2 - Left joins for enrichment:
  Options considered: Inner join, left join
  Choice: Left join
  Reason: Preserve all listings even if no reviews/calendar data
  Trade-off: Introduces NULLs but avoids data loss

DECISION 3 - Price outlier threshold of ฿10,000:
  Options considered: IQR method, fixed threshold, percentile-based
  Choice: Fixed threshold ฿10,000/night
  Reason: Reviewed data manually — prices above this are likely errors
  Trade-off: May exclude genuine luxury properties

DECISION 4 - In-memory processing:
  Options considered: Chunked loading, Dask, in-memory pandas
  Choice: In-memory pandas
  Reason: Dataset fits comfortably in RAM (~500MB total)
  Trade-off: Not scalable to 10x data size without chunking
""")

print("=== SECTION 03 COMPLETE ===")

✓ Enriched dataset saved: C:\Users\Ishini\Desktop\Expernetic_Assignment\data\enriched_listings.csv
  Rows: 28,204
  Columns: 102

=== ENGINEERING DECISION LOG ===

DECISION 1 - SQLite for SQL layer:
  Options considered: PostgreSQL, DuckDB, SQLite
  Choice: SQLite (in-memory)
  Reason: No installation needed, portable, sufficient for this dataset size
  Trade-off: Cannot handle concurrent writes, not production-grade

DECISION 2 - Left joins for enrichment:
  Options considered: Inner join, left join
  Choice: Left join
  Reason: Preserve all listings even if no reviews/calendar data
  Trade-off: Introduces NULLs but avoids data loss

DECISION 3 - Price outlier threshold of ฿10,000:
  Options considered: IQR method, fixed threshold, percentile-based
  Choice: Fixed threshold ฿10,000/night
  Reason: Reviewed data manually — prices above this are likely errors
  Trade-off: May exclude genuine luxury properties

DECISION 4 - In-memory processing:
  Options considered: Chunked loading, Das